# 06 · Tier 1 — Cold-start upgrades

Examine **Tier 1** implementations on the cross-regime slice (warm + cold-item + cold-user):

1. **Review-text enrichment** — aggregate Yelp reviews into `items.text`
2. **Content-based retrieval** — enriched text vs name+categories only
3. **Extended ranker features** — `content_score`, retriever-source flags, activity bucket, item age
4. **Unified two-stage** — `MultiRetriever` → LightGBM with extended features

Prerequisites: cross-regime data in `data/processed/` (see `scripts/make_crossregime.py`).
Run `python scripts/enrich_item_text.py` for review snippets on real Yelp.

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.recsys.config import settings
from src.recsys.data import load_dataset
from src.recsys.models import (
    ContentBasedRecommender,
    MultiRetriever,
    PopularityRecommender,
    SocialRecommender,
    TwoStageRecommender,
    TwoTowerRecommender,
)
from scripts.benchmark import (
    _score_recs,
    build_slices,
    build_unified_two_stage,
    slice_ground_truth,
)

ds = load_dataset()
train, test_pos, cold_items, cold_users = build_slices(ds, cutoff_quantile=0.9)
warm_gt, ci_gt, cu_gt = slice_ground_truth(test_pos, cold_items, cold_users)
users = sorted(set(warm_gt) | set(ci_gt) | set(cu_gt))
k = settings.top_k

print(ds.summary())
print(f"cold items={len(cold_items):,} cold users={len(cold_users):,}")
print(f"slice users: warm={len(warm_gt):,} cold_item={len(ci_gt):,} cold_user={len(cu_gt):,}")

## 1. Review-text enrichment

Inspect enriched `items.text` (name + categories + review snippets). Compare text length before/after enrichment if you have a raw copy.

In [ ]:
items = ds.items.copy()
items["text_len"] = items["text"].fillna("").astype(str).str.len()
print(items[[settings.item_col, "name", "text_len"]].describe())
print("\nSample enriched text:")
print(items.loc[items["text_len"].idxmax(), "text"][:500])

## 2. Content-based retrieval (cold-item signal)

Pure content CF uses item text embeddings. Should lift **cold-item** slice vs collaborative models.

In [ ]:
content = ContentBasedRecommender(items=ds.items)
content.fit(train)
recs = content.recommend(users, k=k)
content_metrics = _score_recs(recs, test_pos, warm_gt, ci_gt, cu_gt, k)
pd.Series(content_metrics, name="content_based")

## 3. Extended ranker features

Build the feature dict the unified ranker consumes: content scores, retriever provenance, activity bucket, item age.

In [ ]:
from src.recsys.models.ranker_features import ExtendedRankerFeatures

sample_user = next(iter(warm_gt))
sample_items = list(warm_gt[sample_user])[:5]
candidates = {sample_user: sample_items}

builder = ExtendedRankerFeatures(train)
feats = builder.build(candidates, content_scores=content.score_candidates(candidates))
pd.DataFrame(feats[sample_user]).T

## 4. Unified two-stage (Tier 1 benchmark architecture)

Compare baseline two-stage (two-tower only) vs **unified** pipeline with social + extended features.
Notebook uses fewer epochs than `scripts/benchmark.py` for speed — scale up for headline numbers.

In [ ]:
NB_EPOCHS = 6  # benchmark uses 10–15; increase for final runs

two_stage = TwoStageRecommender(
    TwoTowerRecommender(dim=64, epochs=NB_EPOCHS),
    candidate_n=200,
    use_social=False,
    use_extended_features=False,
)
unified = build_unified_two_stage(ds, use_sasrec=False)
unified.retriever.retrievers[0] = TwoTowerRecommender(dim=64, epochs=NB_EPOCHS)

rows = []
for name, model in [("two_stage", two_stage), ("two_stage_unified", unified)]:
    model.fit(train)
    recs = model.recommend(users, k=k)
    m = _score_recs(recs, test_pos, warm_gt, ci_gt, cu_gt, k)
    rows.append({"model": name, **m})

tier1_df = pd.DataFrame(rows).set_index("model")
tier1_df

In [ ]:
cols = [f"overall_r@{k}", f"warm_r@{k}", f"cold_item_r@{k}", f"cold_user_r@{k}"]
ax = tier1_df[cols].plot.bar(figsize=(10, 4), rot=0, title="Tier 1: four-view recall")
ax.set_ylabel(f"recall@{k}"); ax.legend(bbox_to_anchor=(1.02, 1)); ax.figure.tight_layout()

## 5. Ranker feature importance (unified)

Which extended features drive re-ranking? NaN values indicate sklearn fallback (no LightGBM native importances).

In [ ]:
try:
    imp = unified.ranker.feature_importance()
    imp_s = pd.Series(imp).sort_values(ascending=False)
    imp_s.head(15).plot.barh(figsize=(8, 5), title="Ranker feature importance")
    plt.gca().invert_yaxis(); plt.tight_layout()
except Exception as e:
    print(f"Could not read importances: {e}")

## Next steps

- Full numbers: `python scripts/benchmark.py --mode fast`
- Tier 2 graph + multimodal: **`07_tier2_graph_multimodal.ipynb`**
- See `docs/techniques.md` § Tier 1 for interview talking points.